# Scanner Synthetic Evaluation

Comparison of scanners against synthetic data with a known positive rate.

This notebook is the synthetic-data analog of `scanner_validation.ipynb`. The
key differences:

- There is no human validation file. Synthetic data is assumed to have a
  100% violation rate — every transcript should be flagged.
- Eval files are not always grouped by benchmark name. Multiple eval files
  may share a `transcript_task_set` but use different synthetic-data
  approaches. Display grouping is therefore driven by an explicit
  `eval_file_groups` map in the config.

Results are saved out to a subdirectory under `analysis/results/`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/jeffm/projects/scanner_evaluation")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from analysis.scan_utils import load_scan_results
from analysis.analysis_utils import (
    GRADE_LEVELS,
    SCORE_COLORS,
    shorten_model,
    violation_rate,
    confusion_matrix,
    quadratic_weighted_kappa,
    bootstrap_threshold_metrics,
    bootstrap_kappa,
    format_metric_ci,
    format_metric_ci_columns,
    format_numeric_columns,
    draw_cm,
    make_savers,
)

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)

Run-specific values (target scanner, split, threshold, scan_id filters,
results subdirectory, and the `eval_file_groups` mapping) are loaded from a
YAML file under `analysis/configs/`. Point `CONFIG_PATH` at a different file
to switch runs. Display invariants (model aliases, grade colors) are kept
inline below.

In [ ]:
import yaml

# Per-run config. Swap this path to analyze a different scan.
CONFIG_PATH = PROJECT_ROOT / "analysis" / "configs" / "ground_truth_access_dev_synthetic.yaml"

with open(CONFIG_PATH) as f:
    _cfg = yaml.safe_load(f)

TARGET_SCANNER: str = _cfg["target_scanner"]
SPLIT: str = _cfg["split"]
SCANNER_KEY: str | None = _cfg.get("scanner_key")
VIOLATION_THRESHOLD: int = _cfg["violation_threshold"]
INCLUDE_SCAN_IDS: list[str] = list(_cfg.get("include_scan_ids") or [])
EXCLUDE_SCAN_IDS: list[str] = list(_cfg.get("exclude_scan_ids") or [])
EVAL_FILE_GROUPS: dict[str, str] = dict(_cfg.get("eval_file_groups") or {})

display(Markdown(f"# Scanner Synthetic Evaluation: {TARGET_SCANNER}"))
display(Markdown(f"## {TARGET_SCANNER} — Configuration"))

SCAN_RESULTS_DIR = PROJECT_ROOT / "evals" / "scans" / TARGET_SCANNER / SPLIT / "scan-results"

_results_subdir = _cfg.get("results_subdir")
RESULTS_DIR: Path | None = (
    PROJECT_ROOT / "analysis" / "results" / TARGET_SCANNER / SPLIT / _results_subdir
    if _results_subdir
    else None
)
if RESULTS_DIR is not None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

save_fig, save_table = make_savers(RESULTS_DIR)

# Synthetic data assumes every transcript is a violation.
ASSUMED_POSITIVE_RATE = 1.0

print(f"CONFIG         = {CONFIG_PATH.relative_to(PROJECT_ROOT)}")
print(f"TARGET_SCANNER = {TARGET_SCANNER}")
print(f"SPLIT          = {SPLIT}")
print(f"THRESHOLD      = {VIOLATION_THRESHOLD}")
print(f"SCAN_RESULTS   = {SCAN_RESULTS_DIR}")
print(f"RESULTS_DIR    = {RESULTS_DIR if RESULTS_DIR else '(not saving outputs)'}")
print(f"EVAL_FILE_GROUPS: {len(EVAL_FILE_GROUPS)} mapping(s)")

`load_scan_results` discovers every `scan_id=*` directory, reads each parquet,
and attaches scanner metadata. We then attach the eval-file basename and the
configured group label.

In [ ]:
raw_scans = load_scan_results(SCAN_RESULTS_DIR)
print(f"Loaded {len(raw_scans):,} rows from {raw_scans['scanner_source'].nunique()} scan_id(s).")
present_keys = sorted(raw_scans["scanner_key"].dropna().unique())
print(f"Scanner keys present: {present_keys}")

if SCANNER_KEY is not None:
    resolved_scanner_key = SCANNER_KEY
elif len(present_keys) == 1:
    resolved_scanner_key = present_keys[0]
    if resolved_scanner_key != TARGET_SCANNER:
        print(f"⚠ Directory name '{TARGET_SCANNER}' differs from parquet scanner_key "
              f"'{resolved_scanner_key}'. Using '{resolved_scanner_key}'.")
else:
    raise RuntimeError(
        f"Multiple scanner_keys present ({present_keys}). Set SCANNER_KEY explicitly."
    )

display(Markdown(f"## {resolved_scanner_key} — Load scan results"))

scans = raw_scans.copy()
scans["scan_id"] = scans["scanner_source"].str.removeprefix("scan_id=")
scans["eval_file"] = scans["transcript_source_uri"].fillna("").map(
    lambda p: Path(p).name if p else None
)
scans["scanner_model"] = scans["scanner_model"].map(shorten_model)
scans["eval_generation_model"] = scans["transcript_model"].map(shorten_model)
scans["scanner_label"] = scans["scanner_model"].fillna("(no model)").astype(str) + " · " + scans["scan_id"].str[:6]

scans = scans[scans["scanner_key"] == resolved_scanner_key].copy()
if INCLUDE_SCAN_IDS:
    scans = scans[scans["scan_id"].isin(INCLUDE_SCAN_IDS)].copy()
if EXCLUDE_SCAN_IDS:
    scans = scans[~scans["scan_id"].isin(EXCLUDE_SCAN_IDS)].copy()

# Apply eval_file_groups mapping; eval files not listed fall back to a
# benchmark · eval_file label so they still appear in the plots.
def _group_label(row):
    ef = row["eval_file"]
    if ef in EVAL_FILE_GROUPS:
        return EVAL_FILE_GROUPS[ef]
    bench = row.get("transcript_task_set")
    if isinstance(bench, str) and bench:
        return f"{bench} · {ef[:8] if ef else '?'}"
    return ef or "(unknown)"

scans["group_label"] = scans.apply(_group_label, axis=1)

scan_id_order = (
    scans.sort_values("scan_timestamp")
    .drop_duplicates("scan_id")["scan_id"]
    .tolist()
)
scanner_label_by_scan_id = (
    scans.drop_duplicates("scan_id").set_index("scan_id")["scanner_label"].to_dict()
)

# Group ordering follows the YAML ordering when configured, then anything
# else. Dedupe while preserving first-seen order so that multiple eval files
# sharing a label collapse into a single group entry.
_present_labels = set(scans["group_label"])
configured_order = list(dict.fromkeys(
    v for v in EVAL_FILE_GROUPS.values() if v in _present_labels
))
extra_groups = sorted(g for g in _present_labels if g not in configured_order)
group_order = configured_order + extra_groups

print(f"After filtering: {len(scans):,} rows · {len(scan_id_order)} scan_id(s) · {len(group_order)} group(s)")
print()
display(
    scans.drop_duplicates("scan_id")[
        ["scan_id", "scanner_label", "scanner_model", "scan_timestamp"]
    ].sort_values("scan_timestamp").reset_index(drop=True)
)

Per-group overview: count of scanned transcripts, eval-generation model, and
scanner detected violation rate (score >= threshold). Synthetic data target
positive rate is always 100%.

In [ ]:
display(Markdown(f"## {resolved_scanner_key} — Per-group overview"))

overview_rows = []
for (sid, grp), group in scans.groupby(["scan_id", "group_label"], dropna=False):
    overview_rows.append({
        "scan_id": sid,
        "scanner_label": scanner_label_by_scan_id.get(sid, sid),
        "group_label": grp,
        "eval_generation_model": group["eval_generation_model"].dropna().iloc[0]
            if group["eval_generation_model"].notna().any() else None,
        "eval_file": ", ".join(sorted(group["eval_file"].dropna().unique())),
        "n_scanned": len(group),
        "detected_violation_rate": violation_rate(group["value_num"], VIOLATION_THRESHOLD),
        "target_positive_rate": ASSUMED_POSITIVE_RATE,
    })

# Order rows by configured group order, then by scan_id.
overview = pd.DataFrame(overview_rows)
overview["_grp_idx"] = overview["group_label"].map(
    {g: i for i, g in enumerate(group_order)}
).fillna(len(group_order))
overview = overview.sort_values(["scan_id", "_grp_idx"]).drop(columns="_grp_idx").reset_index(drop=True)
save_table(overview, "overview")

display_ov = format_numeric_columns(
    overview, ["detected_violation_rate", "target_positive_rate"],
    decimals=1, as_percent=True,
)
save_table(display_ov, "overview_formatted")
display(display_ov)

### Grade distribution per group, per scan_id

Stacked bars over `group_label`. One subplot per scan_id.

In [ ]:
display(Markdown(f"## {resolved_scanner_key} — Grade distributions"))

def _grade_distribution(df: pd.DataFrame, group_col: str, group_values: list[str]):
    n_per = []
    stacks: dict[int, list[float]] = {g: [] for g in GRADE_LEVELS}
    grades_present_local = sorted(df["value_num"].dropna().astype(int).unique())
    for g in group_values:
        sub = df[df[group_col] == g]["value_num"].dropna()
        n_per.append(int(len(sub)))
        total = len(sub)
        for grade in GRADE_LEVELS:
            stacks[grade].append((sub.astype(int) == grade).sum() / total if total else 0.0)
    return n_per, grades_present_local, stacks


def _plot_stacked_bars(ax, labels, n_per, stacks, all_grades, title):
    x = np.arange(len(labels))
    bottom = np.zeros(len(labels))
    for grade in reversed(all_grades):
        proportions = stacks.get(grade, [0] * len(labels))
        ax.bar(x, proportions, bottom=bottom, label=str(grade),
               color=SCORE_COLORS.get(grade, "#999999"))
        bottom += np.array(proportions)
    for xi, n in zip(x, n_per):
        ax.text(xi, 0.92, f"n={n}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_title(title, fontsize=10)


if scans.empty or not scan_id_order or not group_order:
    print("No scans to plot.")
else:
    # Drop grade 0 from the stacked bars so each bar reads as "share at grade ≥1".
    all_grades = [g for g in sorted(scans["value_num"].dropna().astype(int).unique()) if g >= 1]
    n_runs = len(scan_id_order)
    fig, axes = plt.subplots(
        1, n_runs,
        figsize=(max(4, 0.75 * len(group_order) * n_runs), 4.0),
        sharey=True,
        squeeze=False,
    )
    for ax, sid in zip(axes[0], scan_id_order):
        sub = scans[scans["scan_id"] == sid]
        n_per, _grades, stacks = _grade_distribution(sub, "group_label", group_order)
        _plot_stacked_bars(
            ax, group_order, n_per, stacks, all_grades,
            title=scanner_label_by_scan_id.get(sid, sid),
        )
    axes[0][0].set_ylabel("Proportion")
    axes[0][-1].legend(title="Grade", loc="center left", bbox_to_anchor=(1.01, 0.5))
    fig.suptitle(f"{resolved_scanner_key} — Grade Distribution by Group", y=0.98)
    fig.tight_layout()
    save_fig(fig, "grade_distribution")
    plt.show()

### Detected violation rate per group

Bar = scanner detected rate. Reference line = assumed synthetic positive rate
(100%). The gap is the false-negative rate (transcripts the scanner missed).

In [ ]:
display(Markdown(f"## {resolved_scanner_key} — Detected violation rate"))

if scans.empty or not group_order:
    print("No scans to plot.")
else:
    n_runs = len(scan_id_order)
    fig, axes = plt.subplots(
        1, n_runs,
        figsize=(max(5, 1.3 * len(group_order) * n_runs), 4.0),
        sharey=True,
        squeeze=False,
    )
    for ax, sid in zip(axes[0], scan_id_order):
        sub = scans[scans["scan_id"] == sid]
        rates = []
        ns = []
        for g in group_order:
            cell = sub[sub["group_label"] == g]["value_num"]
            n = int(pd.to_numeric(cell, errors="coerce").notna().sum())
            r = violation_rate(cell, VIOLATION_THRESHOLD) if n else float("nan")
            rates.append(r if pd.notna(r) else 0.0)
            ns.append(n)
        x = np.arange(len(group_order))
        ax.bar(x, rates, color="#d1495b", label="Detected")
        ax.axhline(ASSUMED_POSITIVE_RATE, color="#333333", linestyle="--",
                   linewidth=1, label=f"Target ({ASSUMED_POSITIVE_RATE:.0%})")
        for xi, n in zip(x, ns):
            ax.text(xi, 0.02, f"n={n}", ha="center", va="bottom",
                    fontsize=8, color="white")
        ax.set_xticks(x)
        ax.set_xticklabels(group_order, rotation=45, ha="right", fontsize=9)
        ax.set_ylim(0, 1.05)
        ax.set_title(scanner_label_by_scan_id.get(sid, sid), fontsize=10)
    axes[0][0].set_ylabel(f"Violation rate (score ≥ {VIOLATION_THRESHOLD})")
    axes[0][-1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
    fig.suptitle(f"{resolved_scanner_key} — Detected Violation Rate by Group", y=0.98)
    fig.tight_layout()
    save_fig(fig, "detected_violation_rate")
    plt.show()

### Performance metrics vs synthetic ground truth

Every synthetic transcript is assumed positive, so:

- `sensitivity` = fraction of transcripts the scanner correctly flagged
- `specificity` is undefined (no negatives in synthetic data)
- `accuracy` collapses to `sensitivity` for this dataset

In [ ]:
display(Markdown(f"## {resolved_scanner_key} — Performance vs synthetic target"))

# Synthetic data: every transcript is assumed positive, so target is a
# constant array of max grade and only sensitivity / accuracy are meaningful.
SYNTH_CI_COLS = ["sensitivity", "accuracy"]
_synth_target_grade = max(GRADE_LEVELS)


def _synth_metrics_row(scores: pd.Series) -> dict:
    n = int(len(scores))
    pred_int = scores.astype(int).to_numpy()
    target_int = np.full(n, _synth_target_grade, dtype=int)
    flagged = int((pred_int >= VIOLATION_THRESHOLD).sum())
    base = {
        "n": n,
        "tp": flagged,
        "fn": n - flagged,
        "sensitivity": flagged / n,
        "accuracy": flagged / n,
    }
    ci = bootstrap_threshold_metrics(
        target_int, pred_int, threshold=VIOLATION_THRESHOLD,
    )
    for k in SYNTH_CI_COLS:
        lo, hi = ci[k]
        base[f"{k}_lo"] = lo
        base[f"{k}_hi"] = hi
    return base


metrics_rows = []
for sid in scan_id_order:
    sub = scans[scans["scan_id"] == sid]
    for g in group_order:
        cell = sub[sub["group_label"] == g]
        scores = pd.to_numeric(cell["value_num"], errors="coerce").dropna()
        if scores.empty:
            continue
        metrics_rows.append({
            "scan_id": sid,
            "scanner_label": scanner_label_by_scan_id.get(sid, sid),
            "group_label": g,
            **_synth_metrics_row(scores),
        })

# Pooled row across all groups, per scan_id.
for sid in scan_id_order:
    sub = scans[scans["scan_id"] == sid]
    scores = pd.to_numeric(sub["value_num"], errors="coerce").dropna()
    if scores.empty:
        continue
    metrics_rows.append({
        "scan_id": sid,
        "scanner_label": scanner_label_by_scan_id.get(sid, sid),
        "group_label": "ALL (pooled)",
        **_synth_metrics_row(scores),
    })

if not metrics_rows:
    print("No data to compute metrics.")
else:
    metrics_df = pd.DataFrame(metrics_rows)
    save_table(metrics_df, "metrics")

    display_metrics = format_metric_ci_columns(metrics_df, SYNTH_CI_COLS)
    save_table(display_metrics, "metrics_formatted")
    display(display_metrics)

### False negatives (missed violations)

Per group, lists transcripts the scanner did NOT flag (`score < threshold`)
even though synthetic ground truth says they should be positive. Useful for
opening individual transcripts to inspect why the scanner missed them.

In [ ]:
display(Markdown(f"## {resolved_scanner_key} — False negatives"))

fn_cols = ["transcript_id", "scan_id", "scanner_label", "group_label",
           "eval_file", "value_num"]
fn_cols = [c for c in fn_cols if c in scans.columns]

scans_scored = scans.dropna(subset=["value_num"]).copy()
scans_scored["value_num"] = pd.to_numeric(scans_scored["value_num"], errors="coerce")
scans_scored = scans_scored.dropna(subset=["value_num"])

if scans_scored.empty:
    print("No scored transcripts.")
else:
    fn_all = scans_scored[scans_scored["value_num"] < VIOLATION_THRESHOLD]
    save_table(fn_all[fn_cols].rename(columns={"value_num": "scanner_grade"}),
               "false_negatives")
    for sid in scan_id_order:
        for g in group_order:
            cell = scans_scored[
                (scans_scored["scan_id"] == sid) & (scans_scored["group_label"] == g)
            ]
            if cell.empty:
                continue
            misses = cell[cell["value_num"] < VIOLATION_THRESHOLD]
            print("=" * 70)
            print(f"{scanner_label_by_scan_id.get(sid, sid)}  |  {g}")
            print(f"  missed: {len(misses)} / {len(cell)} "
                  f"({len(misses) / len(cell):.1%})")
            if misses.empty:
                print("  perfect detection")
                continue
            display(
                misses[fn_cols]
                .rename(columns={"value_num": "scanner_grade"})
                .reset_index(drop=True)
            )

### Pairwise scanner agreement (when ≥2 scan_ids)

For every pair of scan_ids that share transcripts, plot a 4×4 agreement
matrix and report quadratic-weighted κ. Skipped when only one scan_id is
loaded.

In [ ]:
display(Markdown(f"## {resolved_scanner_key} — Scanner-vs-scanner agreement"))

if len(scan_id_order) < 2:
    print(f"Only {len(scan_id_order)} scan_id loaded — skipping pairwise agreement.")
else:
    pivot = (
        scans.dropna(subset=["value_num"])
        .pivot_table(
            index="transcript_id",
            columns="scan_id",
            values="value_num",
            aggfunc="first",
        )
    )
    pairs = list(itertools.combinations(scan_id_order, 2))
    n_pairs = len(pairs)
    n_cols = min(3, n_pairs)
    n_rows = (n_pairs + n_cols - 1) // n_cols
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4.4 * n_cols, 4.0 * n_rows),
        squeeze=False,
    )
    pair_rows = []
    for idx, (sid_a, sid_b) in enumerate(pairs):
        ax = axes[idx // n_cols][idx % n_cols]
        if sid_a not in pivot.columns or sid_b not in pivot.columns:
            ax.axis("off")
            continue
        joint = pivot[[sid_a, sid_b]].dropna()
        if joint.empty:
            ax.axis("off")
            ax.set_title(f"{sid_a[:6]} vs {sid_b[:6]}\n(no overlap)", fontsize=9)
            continue
        ga = joint[sid_a].astype(int).to_numpy()
        gb = joint[sid_b].astype(int).to_numpy()
        cm = confusion_matrix(ga, gb, GRADE_LEVELS)
        kappa = quadratic_weighted_kappa(cm)
        kappa_lo, kappa_hi = bootstrap_kappa(ga, gb)
        n = int(cm.sum())
        agree = int(np.trace(cm))
        kappa_str = f"{kappa:.3f}" if not np.isnan(kappa) else "—"
        draw_cm(
            ax, cm,
            xlabel=scanner_label_by_scan_id.get(sid_b, sid_b),
            ylabel=scanner_label_by_scan_id.get(sid_a, sid_a),
            title=f"{sid_a[:6]} vs {sid_b[:6]}\nn={n}, agree={agree/n:.1%}, qwκ={kappa_str}",
        )
        pair_rows.append({
            "scan_a": sid_a,
            "scan_b": sid_b,
            "n_paired": n,
            "agree_rate": agree / n if n else float("nan"),
            "mean_abs_diff": float(np.abs(ga - gb).mean()) if n else float("nan"),
            "quadratic_weighted_kappa": kappa,
            "quadratic_weighted_kappa_lo": kappa_lo,
            "quadratic_weighted_kappa_hi": kappa_hi,
        })
    for idx in range(n_pairs, n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].axis("off")
    fig.suptitle(f"{resolved_scanner_key} — Pairwise Agreement", y=0.95)
    fig.tight_layout()
    save_fig(fig, "scanner_agreement")
    plt.show()

    if pair_rows:
        pair_df = pd.DataFrame(pair_rows)
        save_table(pair_df, "pairwise_agreement")

        display_pair = format_metric_ci_columns(pair_df, ["quadratic_weighted_kappa"])
        display_pair = format_numeric_columns(
            display_pair, ["agree_rate"], decimals=1, as_percent=True,
        )
        display_pair = format_numeric_columns(
            display_pair, ["mean_abs_diff"], decimals=2,
        )
        save_table(display_pair, "pairwise_agreement_formatted")
        display(display_pair)